In [2]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from surprise import SVD, Dataset, Reader
from surprise import accuracy
from collections import defaultdict

train_df = pd.read_csv('../data/processed/train.csv')
test_df  = pd.read_csv('../data/processed/test.csv')

# 영화 메타데이터 로드
movies_df = pd.read_csv('../data/raw/u.item', sep='|', encoding='latin-1',
                        names=['movieId','title','release_date','video_release',
                               'imdb_url','unknown','Action','Adventure','Animation',
                               'Children','Comedy','Crime','Documentary','Drama',
                               'Fantasy','Film-Noir','Horror','Musical','Mystery',
                               'Romance','Sci-Fi','Thriller','War','Western'])

genre_cols = ['Action','Adventure','Animation','Children','Comedy','Crime',
              'Documentary','Drama','Fantasy','Film-Noir','Horror','Musical',
              'Mystery','Romance','Sci-Fi','Thriller','War','Western']

print(f"movies: {len(movies_df)}개")
print(movies_df[['movieId','title'] + genre_cols[:5]].head())

movies: 1682개
   movieId              title  Action  Adventure  Animation  Children  Comedy
0        1   Toy Story (1995)       0          0          1         1       1
1        2   GoldenEye (1995)       1          1          0         0       0
2        3  Four Rooms (1995)       0          0          0         0       0
3        4  Get Shorty (1995)       1          0          0         0       1
4        5     Copycat (1995)       0          0          0         0       0


In [3]:
# 장르 컬럼을 텍스트로 변환 (TF-IDF 입력용)
movies_df['genre_text'] = movies_df[genre_cols].apply(
    lambda row: ' '.join([genre_cols[i] for i, v in enumerate(row) if v == 1]),
    axis=1
)

print("장르 텍스트 예시:")
print(movies_df[['movieId','title','genre_text']].head())

# TF-IDF 벡터화
tfidf = TfidfVectorizer()
tfidf_matrix = tfidf.fit_transform(movies_df['genre_text'])

# 코사인 유사도 행렬 계산
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

# movieId → 행렬 인덱스 매핑
movie_ids = movies_df['movieId'].tolist()
id_to_idx = {mid: idx for idx, mid in enumerate(movie_ids)}

print(f"\nTF-IDF 행렬 크기: {tfidf_matrix.shape}")
print(f"코사인 유사도 행렬 크기: {cosine_sim.shape}")
print("CBF 구축 완료")

장르 텍스트 예시:
   movieId              title                 genre_text
0        1   Toy Story (1995)  Animation Children Comedy
1        2   GoldenEye (1995)  Action Adventure Thriller
2        3  Four Rooms (1995)                   Thriller
3        4  Get Shorty (1995)        Action Comedy Drama
4        5     Copycat (1995)       Crime Drama Thriller

TF-IDF 행렬 크기: (1682, 20)
코사인 유사도 행렬 크기: (1682, 1682)
CBF 구축 완료


In [4]:
def cbf_predict(user_id, movie_id, train_df, cosine_sim, id_to_idx, k=20):
    """
    특정 유저가 본 영화들과 target 영화의 유사도 기반으로 예측 평점 계산
    """
    if movie_id not in id_to_idx:
        return None

    target_idx = id_to_idx[movie_id]

    # 해당 유저의 train 평점 가져오기
    user_ratings = train_df[train_df['userId'] == user_id][['movieId','rating']]

    if len(user_ratings) == 0:
        return None

    # 유저가 평가한 영화들과 target 영화의 유사도 계산
    sims, ratings = [], []
    for _, row in user_ratings.iterrows():
        mid = row['movieId']
        if mid in id_to_idx:
            sim = cosine_sim[target_idx][id_to_idx[mid]]
            if sim > 0:
                sims.append(sim)
                ratings.append(row['rating'])

    if len(sims) == 0:
        return train_df['rating'].mean()

    # 상위 k개 유사 영화의 가중 평균
    sim_rating = sorted(zip(sims, ratings), reverse=True)[:k]
    s, r = zip(*sim_rating)
    return np.dot(s, r) / np.sum(s)

# test 전체에 대해 CBF 예측
print("CBF 예측 중... (2~3분 소요)")

cbf_preds = []
for _, row in test_df.iterrows():
    pred = cbf_predict(row['userId'], row['movieId'],
                       train_df, cosine_sim, id_to_idx)
    if pred is None:
        pred = train_df['rating'].mean()
    pred = np.clip(pred, 1, 5)
    cbf_preds.append((row['userId'], row['movieId'], row['rating'], pred, None))

# RMSE 계산
from surprise import accuracy
from surprise.prediction_algorithms.predictions import Prediction

surprise_preds = [Prediction(uid, iid, r_ui, est, None)
                  for uid, iid, r_ui, est, _ in cbf_preds]

rmse_cbf = accuracy.rmse(surprise_preds, verbose=False)
mae_cbf  = accuracy.mae(surprise_preds,  verbose=False)

print(f"CBF RMSE: {rmse_cbf:.4f}")
print(f"CBF MAE:  {mae_cbf:.4f}")

CBF 예측 중... (2~3분 소요)
CBF RMSE: 1.1232
CBF MAE:  0.9424


In [5]:
# TemporalSVD_v2 클래스와 함수 다시 정의 (이 노트북에서 사용하기 위해)
def apply_temporal_weight(df, lam):
    t_max  = df['timestamp'].max()
    t_min  = df['timestamp'].min()
    t_half = (t_max - t_min) / 2
    weights = np.exp(-lam * (t_max - df['timestamp']) / (t_half + 1))
    df = df.copy()
    df['weight'] = weights
    return df

reader = Reader(rating_scale=(1, 5))

# 최적 λ=0.1로 TemporalSVD 학습
weighted_df = apply_temporal_weight(train_df, lam=0.1).copy()
global_mean = weighted_df['rating'].mean()
weighted_df['rating_adjusted'] = (
    weighted_df['weight'] * weighted_df['rating'] +
    (1 - weighted_df['weight']) * global_mean
)
dataset = Dataset.load_from_df(
    weighted_df[['userId','movieId','rating_adjusted']]
    .rename(columns={'rating_adjusted':'rating'}), reader
)
trainset = dataset.build_full_trainset()
tsvd = SVD(n_factors=100, n_epochs=20, lr_all=0.005, reg_all=0.02, random_state=42)
tsvd.fit(trainset)

testset = [(r['userId'], r['movieId'], r['rating']) for _, r in test_df.iterrows()]
tsvd_preds = tsvd.test(testset)
tsvd_pred_dict = {(p.uid, p.iid): p.est for p in tsvd_preds}
cbf_pred_dict  = {(p.uid, p.iid): p.est for p in surprise_preds}

# α 탐색: α·TemporalSVD + (1-α)·CBF
alpha_list = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
hybrid_results = []

print("α 탐색 시작\n")
for alpha in alpha_list:
    hybrid_preds = []
    for _, row in test_df.iterrows():
        key = (row['userId'], row['movieId'])
        svd_est = tsvd_pred_dict.get(key, global_mean)
        cbf_est = cbf_pred_dict.get(key, global_mean)
        hybrid_est = alpha * svd_est + (1 - alpha) * cbf_est
        hybrid_est = np.clip(hybrid_est, 1, 5)
        hybrid_preds.append(Prediction(key[0], key[1], row['rating'], hybrid_est, None))

    rmse = accuracy.rmse(hybrid_preds, verbose=False)
    mae  = accuracy.mae(hybrid_preds,  verbose=False)
    hybrid_results.append({'alpha': alpha, 'RMSE': round(rmse,4), 'MAE': round(mae,4)})
    print(f"α={alpha} → RMSE: {rmse:.4f}  MAE: {mae:.4f}")

hybrid_df = pd.DataFrame(hybrid_results)
best = hybrid_df.loc[hybrid_df['RMSE'].idxmin()]
print(f"\n최적 α: {best['alpha']}  (RMSE: {best['RMSE']})")
print(f"TemporalSVD 단독: 1.0294")
print(f"개선폭: {1.0294 - best['RMSE']:.4f}")

α 탐색 시작

α=0.1 → RMSE: 1.1057  MAE: 0.9259
α=0.2 → RMSE: 1.0898  MAE: 0.9096
α=0.3 → RMSE: 1.0756  MAE: 0.8936
α=0.4 → RMSE: 1.0632  MAE: 0.8778
α=0.5 → RMSE: 1.0527  MAE: 0.8623
α=0.6 → RMSE: 1.0441  MAE: 0.8484
α=0.7 → RMSE: 1.0374  MAE: 0.8374
α=0.8 → RMSE: 1.0327  MAE: 0.8292
α=0.9 → RMSE: 1.0300  MAE: 0.8244

최적 α: 0.9  (RMSE: 1.03)
TemporalSVD 단독: 1.0294
개선폭: -0.0006


In [6]:
def precision_recall_ndcg_at_k(predictions, k=10, threshold=3.5):
    user_est_true = defaultdict(list)
    for uid, iid, true_r, est, _ in predictions:
        user_est_true[uid].append((est, true_r))
    precisions, recalls, ndcgs = {}, {}, {}
    for uid, user_ratings in user_est_true.items():
        user_ratings.sort(key=lambda x: x[0], reverse=True)
        top_k = user_ratings[:k]
        n_rel = sum(1 for (_, r) in user_ratings if r >= threshold)
        n_hit = sum(1 for (_, r) in top_k if r >= threshold)
        precisions[uid] = n_hit / k
        recalls[uid]    = n_hit / n_rel if n_rel > 0 else 0
        dcg  = sum((2**(r >= threshold) - 1) / np.log2(i + 2)
                   for i, (_, r) in enumerate(top_k))
        idcg = sum(1 / np.log2(i + 2) for i in range(min(n_rel, k)))
        ndcgs[uid] = dcg / idcg if idcg > 0 else 0
    return (sum(precisions.values()) / len(precisions),
            sum(recalls.values())    / len(recalls),
            sum(ndcgs.values())      / len(ndcgs))

def compute_avg_ild(predictions, movies_df, genre_cols, k=10):
    user_est = defaultdict(list)
    for uid, iid, _, est, _ in predictions:
        user_est[uid].append((est, iid))
    ild_scores = []
    for uid, items in user_est.items():
        items.sort(key=lambda x: x[0], reverse=True)
        top_k_ids = [int(iid) for _, iid in items[:k]]
        valid_ids = [i for i in top_k_ids if i in movies_df.index]
        if len(valid_ids) < 2:
            continue
        vectors = movies_df.loc[valid_ids, genre_cols].values.astype(float)
        n = len(vectors)
        total, count = 0, 0
        for i in range(n):
            for j in range(i+1, n):
                ni, nj = np.linalg.norm(vectors[i]), np.linalg.norm(vectors[j])
                if ni > 0 and nj > 0:
                    total += 1 - np.dot(vectors[i], vectors[j]) / (ni * nj)
                    count += 1
        if count > 0:
            ild_scores.append(total / count)
    return np.mean(ild_scores)

movies_indexed = movies_df.set_index('movieId')

# 최적 α=0.8 하이브리드 predictions 생성
best_alpha = 0.8
hybrid_final = []
for _, row in test_df.iterrows():
    key = (row['userId'], row['movieId'])
    svd_est = tsvd_pred_dict.get(key, global_mean)
    cbf_est = cbf_pred_dict.get(key, global_mean)
    est = np.clip(best_alpha * svd_est + (1 - best_alpha) * cbf_est, 1, 5)
    hybrid_final.append(Prediction(key[0], key[1], row['rating'], est, None))

# 각 모델 지표 계산
models = {
    'SVD_baseline':   surprise_preds,   # 2주차에서 저장한 베이스라인
    'TemporalSVD_v2': tsvd_preds,
    'CBF':            surprise_preds,
    'Hybrid(α=0.8)':  hybrid_final,
}

# SVD baseline predictions 다시 생성
base_data = Dataset.load_from_df(train_df[['userId','movieId','rating']], reader)
base_trainset = base_data.build_full_trainset()
base_svd = SVD(n_factors=100, n_epochs=20, lr_all=0.005, reg_all=0.02, random_state=42)
base_svd.fit(base_trainset)
base_preds = base_svd.test(testset)

rows = []
for name, preds in [('SVD_baseline', base_preds),
                    ('TemporalSVD_v2', tsvd_preds),
                    ('CBF', surprise_preds),
                    ('Hybrid(α=0.8)', hybrid_final)]:
    rmse = accuracy.rmse(preds, verbose=False)
    mae  = accuracy.mae(preds,  verbose=False)
    p, r, n = precision_recall_ndcg_at_k(preds)
    ild = compute_avg_ild(preds, movies_indexed, genre_cols)
    rows.append({'모델': name, 'RMSE': round(rmse,4), 'MAE': round(mae,4),
                 'Precision@10': round(p,4), 'Recall@10': round(r,4),
                 'NDCG@10': round(n,4), 'ILD': round(ild,4)})

final_df = pd.DataFrame(rows)
print("=== 최종 모델 비교표 ===\n")
print(final_df.to_string(index=False))

final_df.to_csv('../results/final_comparison.csv', index=False)
print("\n저장 완료 → results/final_comparison.csv")

=== 최종 모델 비교표 ===

            모델   RMSE    MAE  Precision@10  Recall@10  NDCG@10    ILD
  SVD_baseline 1.0300 0.8214        0.6518     0.4262   0.7823 0.6723
TemporalSVD_v2 1.0294 0.8223        0.6532     0.4271   0.7828 0.6714
           CBF 1.1232 0.9424        0.5193     0.3806   0.6278 0.6657
 Hybrid(α=0.8) 1.0327 0.8292        0.6538     0.4268   0.7862 0.6677

저장 완료 → results/final_comparison.csv
